# 13 — Simvol səviyyəli CNN (PyTorch, CPU)

**Yeganə sual:** neyron şəbəkə də sqlmap boşluq artefaktına düşürmü, normallaşdırma onu da düzəldirmi?

- Model: bayt səviyyəli embedding (32) → üç paralel 1D konvolyusiya (kernel 3/5/7, hər biri 64 filtr, ReLU) → global max-pool → dropout 0.3 → xətti çıxış.
  Kiçik və standart "char-CNN" (Kim 2014 / Zhang et al. 2015 üslubu); hiperparametr axtarışı edilmir.
- Giriş: UTF-8 baytları, maksimum 512 bayt (datasetin 99.9%-i ≤ 441 bayt).
- **Baza**: giriş yalnız kiçik hərfə salınır (TF-IDF-dəki `lowercase=True` ilə eyni; datasetdə təmiz sorğuların 61%-ində, zərərlilərin 2%-ində böyük hərf var — bu ikinci artefaktı kənarda saxlamaq üçün).
- **Normallaşdırma**: `src/normalize.py::normalize` (TF-IDF modelləri ilə eyni funksiya).
- Təlim: Adam (lr 1e-3), batch 128, maksimum 10 epoxa, validation F1-ə görə ən yaxşı epoxa (səbr 3). Hədd 0.5.
- Eyni qrup bölgüsü (`src/experiments.py::make_split`), seed 42–46; `torch.manual_seed(seed)`, 4 CPU thread.
- Qiymətləndirmə 04 / 10 ilə eyni: orijinal test, insan formatı, obfuskasiya (A və B), HttpParams (sızmasız) və artefakt inyeksiyası testi.

In [ ]:
import sys, time, re
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from experiments import TECH_ORDER, leakage_free, load_clean, load_httpparams, make_split, obfuscate, rates
from normalize import human_format, normalize

RES, FIG = ROOT / "results", ROOT / "figures"
SEEDS = [42, 43, 44, 45, 46]
MAXLEN, BATCH, EPOCHS, PATIENCE = 512, 128, 10, 3
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
print("torch", torch.__version__)
clean, hp_all = load_clean(), load_httpparams()

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, emb=32, filters=64, kernels=(3, 5, 7), dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(257, emb, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(emb, filters, k, padding=k // 2) for k in kernels)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(filters * len(kernels), 1)

    def forward(self, x):
        mask = (x != 0).unsqueeze(1)
        h = self.emb(x).transpose(1, 2)
        pooled = [torch.relu(c(h)).masked_fill(~mask, 0).amax(dim=2) for c in self.convs]
        return self.out(self.drop(torch.cat(pooled, dim=1))).squeeze(1)


def trim(xb):
    # batch-i ən uzun sorğunun uzunluğuna qədər kəsir; padding (0) embedding-i sıfırdır və max-pool-da maskalanır
    return xb[:, :max(int((xb != 0).sum(1).max()), 7)]


def encode(texts):
    arr = np.zeros((len(texts), MAXLEN), dtype=np.int64)
    for i, t in enumerate(texts):
        b = t.encode("utf-8", "surrogatepass")[:MAXLEN]
        arr[i, :len(b)] = np.frombuffer(b, dtype=np.uint8).astype(np.int64) + 1
    return torch.from_numpy(arr)


class CNNDetector:
    def __init__(self, prep):
        self.prep = prep

    def _x(self, xs):
        return encode([self.prep(x) for x in xs])

    def fit(self, train, val, seed):
        torch.manual_seed(seed)
        g = torch.Generator().manual_seed(seed)
        self.model = CharCNN()
        opt = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        lossf = nn.BCEWithLogitsLoss()
        X, y = self._x(train.Query), torch.tensor(train.Label.values, dtype=torch.float32)
        best, best_state, bad, self.history = -1, None, 0, []
        for ep in range(EPOCHS):
            self.model.train()
            perm = torch.randperm(len(X), generator=g)
            for i in range(0, len(X), BATCH):
                idx = perm[i:i + BATCH]
                opt.zero_grad()
                lossf(self.model(trim(X[idx])), y[idx]).backward()
                opt.step()
            f1 = rates(val.Label, self.predict(val.Query))["f1"]
            self.history.append(f1)
            if f1 > best:
                best, bad = f1, 0
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE:
                    break
        self.model.load_state_dict(best_state)
        self.best_val_f1, self.epochs = best, len(self.history)
        return self

    @torch.no_grad()
    def proba(self, xs):
        self.model.eval()
        X = self._x(list(xs))
        return torch.cat([torch.sigmoid(self.model(trim(X[i:i + 1024]))) for i in range(0, len(X), 1024)]).numpy()

    def predict(self, xs):
        return (self.proba(xs) >= 0.5).astype(int)


CONFIGS = {"CNN baza": str.lower, "CNN + normallaşdırma": normalize}

In [ ]:
def sqlmap_spacing(q):
    q = re.sub(r"\s*([=()])\s*", r"  \1  ", q)
    return re.sub(r" {3,}", "  ", q).strip()

rows = []
def add(seed, cfg, metric, value):
    rows.append({"seed": seed, "config": cfg, "metric": metric, "value": float(value)})

for s in SEEDS:
    train, val, test = make_split(clean, s)
    mal = test.loc[test.Label == 1, "Query"]
    obf_df = obfuscate(mal, s)
    test_h = test.assign(Query=test.Query.map(human_format))
    hp = leakage_free(hp_all, train, val)
    hp_mal = hp.loc[hp.Label == 1, "Query"]
    hp_obf = obfuscate(hp_mal, s)
    for cfg, prep in CONFIGS.items():
        t0 = time.perf_counter()
        det = CNNDetector(prep).fit(train, val, s)
        add(s, cfg, "epochs", det.epochs); add(s, cfg, "val_f1", det.best_val_f1)
        for ds, d in (("test", test), ("human", test_h), ("httpparams", hp)):
            r = rates(d.Label, det.predict(d.Query))
            for k in ("f1", "recall", "fpr"):
                add(s, cfg, f"{k}_{ds}", r[k])
        for sc in ("A", "B"):
            add(s, cfg, f"recall_{sc}_original", det.predict(mal.map(unquote) if sc == "A" else mal).mean())
            for t in TECH_ORDER:
                add(s, cfg, f"recall_{sc}_{t}", det.predict(obf_df.loc[obf_df.technique == t, f"input_{sc}"]).mean())
        add(s, cfg, "recall_httpparams_alt_whitespace_A",
            det.predict(hp_obf.loc[hp_obf.technique == "alt_whitespace", "input_A"]).mean())
        missed = hp_mal[det.predict(hp_mal) == 0]
        add(s, cfg, "httpparams_missed", len(missed))
        add(s, cfg, "httpparams_missed_recovered_by_sqlmap_spacing",
            det.predict(missed.map(sqlmap_spacing)).mean() if len(missed) else np.nan)
        print(f"seed {s} {cfg:22s} epochs={det.epochs} valF1={det.best_val_f1:.4f} ({time.perf_counter() - t0:.0f} s)")

cnn = pd.DataFrame(rows)
cnn.to_csv(RES / "13_char_cnn_raw.csv", index=False)

## Nəticələr (orta ± std, 5 seed)

In [ ]:
a = cnn.groupby(["config", "metric"]).value.agg(["mean", "std"])
a.to_csv(RES / "13_char_cnn_mean_std.csv")
fmt = lambda d: (d["mean"] * 100).map("{:.2f}".format) + " ± " + (d["std"] * 100).map("{:.2f}".format)
key_rows = ["f1_test", "recall_test", "fpr_test", "f1_human", "recall_human", "recall_A_original"] + \
           [f"recall_A_{t}" for t in TECH_ORDER] + ["recall_httpparams", "fpr_httpparams", "recall_httpparams_alt_whitespace_A",
                                                    "httpparams_missed_recovered_by_sqlmap_spacing"]
summary = fmt(a).unstack("config").loc[key_rows]
summary

In [ ]:
print(a.xs("httpparams_missed", level="metric"))
print(a.xs("epochs", level="metric"))

### TF-IDF + LR ilə yan-yana (12-dən, eyni seed-lər)
Artefaktın əsas göstəriciləri: insan formatında F1, alternativ whitespace (A) recall-u.

In [ ]:
m12 = pd.read_csv(RES / "12_T4_defense_A_mean_std.csv").set_index(["model", "metric"])
rows_cmp = []
for metric, lab in (("f1_human", "F1, insan formatı"), ("alt_whitespace", "Recall, alt. whitespace (A)"),
                    ("combination", "Recall, kombinasiya (A)")):
    cnn_key = {"f1_human": "f1_human", "alt_whitespace": "recall_A_alt_whitespace",
               "combination": "recall_A_combination"}[metric]
    row = {"göstərici": lab}
    for cfg12, name in (("Baza", "LR baza"), ("Normallaşdırma", "LR + norm.")):
        r = m12.loc[(cfg12, metric)]
        row[name] = f"{r['mean']:.2f} ± {r['std']:.2f}"
    for cfg in CONFIGS:
        r = a.loc[(cfg, cnn_key)]
        row[cfg] = f"{100 * r['mean']:.2f} ± {100 * r['std']:.2f}"
    rows_cmp.append(row)
cmp = pd.DataFrame(rows_cmp).set_index("göstərici")
cmp.to_csv(RES / "13_cnn_vs_lr.csv")
cmp

In [ ]:
order = ["recall_A_original", "recall_A_alt_whitespace", "recall_A_inline_comment_space",
         "recall_A_combination", "recall_human", "recall_httpparams", "recall_httpparams_alt_whitespace_A"]
labels = ["Orijinal", "Alt. whitespace", "Inline şərh", "Kombinasiya", "İnsan formatı", "HttpParams", "HttpParams +\nalt. whitespace"]
fig, ax = plt.subplots(figsize=(9, 4))
x, w = np.arange(len(order)), 0.38
for i, (cfg, col) in enumerate(zip(CONFIGS, ["#4a6fa5", "#d08c3f"])):
    m = a.loc[cfg]["mean"].loc[order] * 100
    sd = a.loc[cfg]["std"].loc[order] * 100
    b = ax.bar(x + (i - 0.5) * w, m, w, yerr=sd, capsize=2, label=cfg, color=col)
    ax.bar_label(b, fmt="%.1f", fontsize=7, padding=2)
ax.set_xticks(x, labels, fontsize=8)
ax.set_ylabel("Recall, % (orta ± std, 5 seed)")
ax.set_ylim(0, 110)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=8)
fig.savefig(FIG / "13_char_cnn_artifact.png", dpi=300, bbox_inches="tight")
plt.show()